In [68]:
import pandas as pd
import numpy as np
import ast

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression

### TF-IDF Version
### Sample movie datase with text features

data_tfidf = pd.read_csv("D:\\AI Models\\archive\\movie_data\\tmdb_5000_movies.csv")
data_tfidf = data_tfidf[["title","overview","keywords"]]
data_tfidf.dropna(inplace = True)
movies_tfidf = pd.DataFrame(data_tfidf)
movies_tfidf

,title,overview,keywords
0,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"":..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na..."
2,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name..."
3,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,..."
4,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 818, ""name"": ""based on novel""}, {""id"":..."
...,...,...,...
4798,El Mariachi,El Mariachi just wants to play his guitar and ...,"[{""id"": 5616, ""name"": ""united states\u2013mexi..."
4799,Newlyweds,A newlywed couple's honeymoon is upended by th...,[]
4800,"Signed, Sealed, Delivered","""Signed, Sealed, Delivered"" introduces a dedic...","[{""id"": 248, ""name"": ""date""}, {""id"": 699, ""nam..."
4801,Shanghai Calling,When ambitious New York attorney Sam is sent t...,[]


In [75]:
def extract_keywords(text):
    try:
        keywords_list = ast.literal_eval(text)
        ###print(keywords_list)
        
        # Extract only the 'name' of each keyword
        extracted = [i['name'] for i in keywords_list]
        
        return extracted   # return as LIST
        
    except:
        return []

movies_tfidf['Movies_keywords'] = movies_tfidf['keywords'].apply(extract_keywords)
movies_tfidf = movies_tfidf.drop('keywords',axis=1)
movies_tfidf

C:\Users\WaqarYounus\AppData\Local\Temp\ipykernel_1280\4135266651.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  movies_tfidf['Movies_keywords'] = movies_tfidf['keywords'].apply(extract_keywords)


,title,overview,Movies_keywords
0,Avatar,"In the 22nd century, a paraplegic Marine is di...","[culture clash, future, space war, space colon..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[ocean, drug abuse, exotic island, east india ..."
2,Spectre,A cryptic message from Bond’s past sends him o...,"[spy, based on novel, secret agent, sequel, mi..."
3,The Dark Knight Rises,Following the death of District Attorney Harve...,"[dc comics, crime fighter, terrorist, secret i..."
4,John Carter,"John Carter is a war-weary, former military ca...","[based on novel, mars, medallion, space travel..."
...,...,...,...
4795,Bang,A young woman in L.A. is having a bad day: she...,"[gang, audition, police fake, homeless, actress]"
4796,Primer,Friends/fledgling entrepreneurs invent a devic...,"[distrust, garage, identity crisis, time trave..."
4798,El Mariachi,El Mariachi just wants to play his guitar and ...,"[united states–mexico barrier, legs, arms, pap..."
4800,"Signed, Sealed, Delivered","""Signed, Sealed, Delivered"" introduces a dedic...","[date, love at first sight, narration, investi..."


In [76]:
### Combine text for TF-IDF
### drop all the empty lists
movies_tfidf = movies_tfidf[movies_tfidf['Movies_keywords'].apply(lambda x: len(x) > 0)]
print(movies_tfidf.shape)

movies_tfidf

(4389, 3)


,title,overview,Movies_keywords
0,Avatar,"In the 22nd century, a paraplegic Marine is di...","[culture clash, future, space war, space colon..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[ocean, drug abuse, exotic island, east india ..."
2,Spectre,A cryptic message from Bond’s past sends him o...,"[spy, based on novel, secret agent, sequel, mi..."
3,The Dark Knight Rises,Following the death of District Attorney Harve...,"[dc comics, crime fighter, terrorist, secret i..."
4,John Carter,"John Carter is a war-weary, former military ca...","[based on novel, mars, medallion, space travel..."
...,...,...,...
4795,Bang,A young woman in L.A. is having a bad day: she...,"[gang, audition, police fake, homeless, actress]"
4796,Primer,Friends/fledgling entrepreneurs invent a devic...,"[distrust, garage, identity crisis, time trave..."
4798,El Mariachi,El Mariachi just wants to play his guitar and ...,"[united states–mexico barrier, legs, arms, pap..."
4800,"Signed, Sealed, Delivered","""Signed, Sealed, Delivered"" introduces a dedic...","[date, love at first sight, narration, investi..."


In [77]:
movies_tfidf['Movies_keywords'] = movies_tfidf['Movies_keywords'].apply(lambda x: " ".join(x))
movies_tfidf

,title,overview,Movies_keywords
0,Avatar,"In the 22nd century, a paraplegic Marine is di...",culture clash future space war space colony so...
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",ocean drug abuse exotic island east india trad...
2,Spectre,A cryptic message from Bond’s past sends him o...,spy based on novel secret agent sequel mi6 bri...
3,The Dark Knight Rises,Following the death of District Attorney Harve...,dc comics crime fighter terrorist secret ident...
4,John Carter,"John Carter is a war-weary, former military ca...",based on novel mars medallion space travel pri...
...,...,...,...
4795,Bang,A young woman in L.A. is having a bad day: she...,gang audition police fake homeless actress
4796,Primer,Friends/fledgling entrepreneurs invent a devic...,distrust garage identity crisis time travel ti...
4798,El Mariachi,El Mariachi just wants to play his guitar and ...,united states–mexico barrier legs arms paper k...
4800,"Signed, Sealed, Delivered","""Signed, Sealed, Delivered"" introduces a dedic...",date love at first sight narration investigati...


In [89]:
movies_tfidf['user_like'] = np.random.randint(0, 2, size=len(movies_tfidf))
movies_tfidf

,title,overview,Movies_keywords,user_like
0,Avatar,"In the 22nd century, a paraplegic Marine is di...",culture clash future space war space colony so...,1
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",ocean drug abuse exotic island east india trad...,1
2,Spectre,A cryptic message from Bond’s past sends him o...,spy based on novel secret agent sequel mi6 bri...,0
3,The Dark Knight Rises,Following the death of District Attorney Harve...,dc comics crime fighter terrorist secret ident...,1
4,John Carter,"John Carter is a war-weary, former military ca...",based on novel mars medallion space travel pri...,1
...,...,...,...,...
4795,Bang,A young woman in L.A. is having a bad day: she...,gang audition police fake homeless actress,0
4796,Primer,Friends/fledgling entrepreneurs invent a devic...,distrust garage identity crisis time travel ti...,0
4798,El Mariachi,El Mariachi just wants to play his guitar and ...,united states–mexico barrier legs arms paper k...,1
4800,"Signed, Sealed, Delivered","""Signed, Sealed, Delivered"" introduces a dedic...",date love at first sight narration investigati...,1


In [81]:

### TF-IDF Vectorization

tfidf = TfidfVectorizer(stop_words='english') 
X_tfidf = tfidf.fit_transform(movies_tfidf['Movies_keywords'])
X_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 48835 stored elements and shape (4389, 7064)>

In [91]:
### User likes/dislikes
### 1 = like, 0 = dislike

user_likes = movies_tfidf['user_like']
user_likes

0       1
1       1
2       0
3       1
4       1
       ..
4795    0
4796    0
4798    1
4800    1
4802    1
Name: user_like, Length: 4389, dtype: int32

In [93]:
feature_names = tfidf.get_feature_names_out()
# Convert sparse matrix to dense and create DataFrame
spare_matrix = pd.DataFrame(X_tfidf.toarray(), columns=feature_names)


In [94]:
### Train Logistic Regression

model_tfidf = LogisticRegression() 
model_tfidf.fit(X_tfidf, user_likes)

LogisticRegression()

In [95]:
### Predict for unseen movies (let's assume all for demo)

pred_probs_tfidf = model_tfidf.predict_proba(X_tfidf)[:,1]
pred_probs_tfidf


array([0.4892791 , 0.66195779, 0.47134443, ..., 0.53808368, 0.61748368,
       0.65381264])

In [96]:
movies_tfidf['like_prob'] = pred_probs_tfidf 
movies_tfidf

,title,overview,Movies_keywords,user_like,like_prob
0,Avatar,"In the 22nd century, a paraplegic Marine is di...",culture clash future space war space colony so...,1,0.489279
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",ocean drug abuse exotic island east india trad...,1,0.661958
2,Spectre,A cryptic message from Bond’s past sends him o...,spy based on novel secret agent sequel mi6 bri...,0,0.471344
3,The Dark Knight Rises,Following the death of District Attorney Harve...,dc comics crime fighter terrorist secret ident...,1,0.595999
4,John Carter,"John Carter is a war-weary, former military ca...",based on novel mars medallion space travel pri...,1,0.644790
...,...,...,...,...,...
4795,Bang,A young woman in L.A. is having a bad day: she...,gang audition police fake homeless actress,0,0.398707
4796,Primer,Friends/fledgling entrepreneurs invent a devic...,distrust garage identity crisis time travel ti...,0,0.396678
4798,El Mariachi,El Mariachi just wants to play his guitar and ...,united states–mexico barrier legs arms paper k...,1,0.538084
4800,"Signed, Sealed, Delivered","""Signed, Sealed, Delivered"" introduces a dedic...",date love at first sight narration investigati...,1,0.617484


In [98]:
recommend_tfidf = movies_tfidf.sort_values(by='like_prob', ascending=False)[['title', 'like_prob']] 
print(recommend_tfidf)

                                         title  like_prob
1776                           The Magic Flute   0.831461
3084                             A Better Life   0.805601
1587              The Curse of the Were-Rabbit   0.776763
3305                       The Blood of Heroes   0.776478
3077                                    Malone   0.774109
...                                        ...        ...
197   Harry Potter and the Philosopher's Stone   0.212781
4624                                 Locker 13   0.208420
2184                            Kung Fu Hustle   0.206082
1417                                Alex Cross   0.200868
2112                                  Bad Moms   0.175809

[4389 rows x 2 columns]


In [110]:
genre_likes = movies_tfidf.groupby('title')['user_like'].mean().sort_values(ascending=False)
genre_likes


title
Æon Flux                            1.0
(500) Days of Summer                1.0
Zack and Miri Make a Porno          1.0
ZMD: Zombies of Mass Destruction    1.0
Youth in Revolt                     1.0
                                   ... 
Year One                            0.0
Y Tu Mamá También                   0.0
X2                                  0.0
X-Men: The Last Stand               0.0
8 Mile                              0.0
Name: user_like, Length: 4386, dtype: float64

## TESTING MORE MOVIES ON THE SAME MODEL


In [99]:
### HERE ARE THE MOVIES (SAMPLE)
movie_data = {
    'movie': [
        # Action (5)
        'War',
        'Pathaan',
        'Extraction',
        'The Dark Knight',
        'Gladiator',

        # Space (2)
        'Ad Astra',
        'Moon',

        # Romantic (4)
        'Kal Ho Naa Ho',
        'Dilwale Dulhania Le Jayenge',
        'Before Sunrise',
        'A Walk to Remember',

        # Other (1)
        '3 Idiots'
    ],

    'overview': [
        # Action
        'Indian agents fight a global threat',
        'Spy returns to stop terrorists',
        'Mercenary rescues a kidnapped boy',
        'Vigilante fights crime in a dark city',
        'Roman general seeks revenge',

        # Space
        'Astronaut travels deep into space',
        'Man works alone on the moon',

        # Romantic
        'Love friendship and life lessons',
        'Love story across cultures',
        'Two strangers fall in love',
        'Young love and illness',

        # Other
        'College life and friendship'
    ],

    'keywords': [
        # Action
        'spy action fight',
        'spy terrorist action',
        'mercenary rescue action',
        'vigilante crime action',
        'war revenge action',

        # Space
        'space astronaut journey',
        'moon isolation space',

        # Romantic
        'love friendship drama',
        'romance family culture',
        'love travel romance',
        'romance illness drama',

        # Other
        'education friendship life'
    ]
}
data_tfidf_2 = pd.DataFrame(movie_data)
data_tfidf_2

,movie,overview,keywords
0,War,Indian agents fight a global threat,spy action fight
1,Pathaan,Spy returns to stop terrorists,spy terrorist action
2,Extraction,Mercenary rescues a kidnapped boy,mercenary rescue action
3,The Dark Knight,Vigilante fights crime in a dark city,vigilante crime action
4,Gladiator,Roman general seeks revenge,war revenge action
5,Ad Astra,Astronaut travels deep into space,space astronaut journey
6,Moon,Man works alone on the moon,moon isolation space
7,Kal Ho Naa Ho,Love friendship and life lessons,love friendship drama
8,Dilwale Dulhania Le Jayenge,Love story across cultures,romance family culture
9,Before Sunrise,Two strangers fall in love,love travel romance


In [100]:
### Combine text for TF-IDF

data_tfidf_2['combined'] = data_tfidf_2['overview'] + ' ' + data_tfidf_2['keywords']
data_tfidf_2

,movie,overview,keywords,combined
0,War,Indian agents fight a global threat,spy action fight,Indian agents fight a global threat spy action...
1,Pathaan,Spy returns to stop terrorists,spy terrorist action,Spy returns to stop terrorists spy terrorist a...
2,Extraction,Mercenary rescues a kidnapped boy,mercenary rescue action,Mercenary rescues a kidnapped boy mercenary re...
3,The Dark Knight,Vigilante fights crime in a dark city,vigilante crime action,Vigilante fights crime in a dark city vigilant...
4,Gladiator,Roman general seeks revenge,war revenge action,Roman general seeks revenge war revenge action
5,Ad Astra,Astronaut travels deep into space,space astronaut journey,Astronaut travels deep into space space astron...
6,Moon,Man works alone on the moon,moon isolation space,Man works alone on the moon moon isolation space
7,Kal Ho Naa Ho,Love friendship and life lessons,love friendship drama,Love friendship and life lessons love friendsh...
8,Dilwale Dulhania Le Jayenge,Love story across cultures,romance family culture,Love story across cultures romance family culture
9,Before Sunrise,Two strangers fall in love,love travel romance,Two strangers fall in love love travel romance


In [101]:

X_new_movies = tfidf.transform(data_tfidf_2['combined'])
X_new_movies

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 59 stored elements and shape (12, 7064)>

In [102]:
pred_probs_tfidf_2 = model_tfidf.predict_proba(X_new_movies)[:,1]
pred_probs_tfidf_2

array([0.33986149, 0.42733194, 0.48227013, 0.42374369, 0.34251477,
       0.50317502, 0.49438994, 0.48613159, 0.40842776, 0.62272893,
       0.58401697, 0.69707274])

In [103]:
data_tfidf_2['like_prob_2'] = pred_probs_tfidf_2 
data_tfidf_2

,movie,overview,keywords,combined,like_prob_2
0,War,Indian agents fight a global threat,spy action fight,Indian agents fight a global threat spy action...,0.339861
1,Pathaan,Spy returns to stop terrorists,spy terrorist action,Spy returns to stop terrorists spy terrorist a...,0.427332
2,Extraction,Mercenary rescues a kidnapped boy,mercenary rescue action,Mercenary rescues a kidnapped boy mercenary re...,0.482270
3,The Dark Knight,Vigilante fights crime in a dark city,vigilante crime action,Vigilante fights crime in a dark city vigilant...,0.423744
4,Gladiator,Roman general seeks revenge,war revenge action,Roman general seeks revenge war revenge action,0.342515
5,Ad Astra,Astronaut travels deep into space,space astronaut journey,Astronaut travels deep into space space astron...,0.503175
6,Moon,Man works alone on the moon,moon isolation space,Man works alone on the moon moon isolation space,0.494390
7,Kal Ho Naa Ho,Love friendship and life lessons,love friendship drama,Love friendship and life lessons love friendsh...,0.486132
8,Dilwale Dulhania Le Jayenge,Love story across cultures,romance family culture,Love story across cultures romance family culture,0.408428
9,Before Sunrise,Two strangers fall in love,love travel romance,Two strangers fall in love love travel romance,0.622729


In [104]:
recommend_tfidf = data_tfidf_2.sort_values(by='like_prob_2', ascending=False)[['movie', 'like_prob_2']] 
print(recommend_tfidf)

                          movie  like_prob_2
11                     3 Idiots     0.697073
9                Before Sunrise     0.622729
10           A Walk to Remember     0.584017
5                      Ad Astra     0.503175
6                          Moon     0.494390
7                 Kal Ho Naa Ho     0.486132
2                    Extraction     0.482270
1                       Pathaan     0.427332
3               The Dark Knight     0.423744
8   Dilwale Dulhania Le Jayenge     0.408428
4                     Gladiator     0.342515
0                           War     0.339861


### TEST SOME MORE DATA 

In [111]:
Movie_data_03 = { 'movie': ['Interstellar', 'Inception', 'The Martian', 'Gravity', 'Titanic', 'Notebook', 'La La Land', 'Avengers', 'John Wick', 'Mad Max'], 'overview': [ 'Space exploration to save humanity', 'Dream within a dream', 'Astronaut stranded on Mars', 'Astronaut in space', 'Love story on a ship', 'Romantic drama', 'Music and love story', 'Superhero fights evil', 'Assassin revenge', 'Post-apocalypse chase' ], 'keywords': [ 'space nasa future', 'dream mind action', 'mars space survival', 'space astronaut', 'ship love tragedy', 'love relationship', 'music love', 'superhero fight', 'assassin revenge', 'post-apocalypse chase' ] }

movies_3 = pd.DataFrame(Movie_data_03)
movies_3

,movie,overview,keywords
0,Interstellar,Space exploration to save humanity,space nasa future
1,Inception,Dream within a dream,dream mind action
2,The Martian,Astronaut stranded on Mars,mars space survival
3,Gravity,Astronaut in space,space astronaut
4,Titanic,Love story on a ship,ship love tragedy
5,Notebook,Romantic drama,love relationship
6,La La Land,Music and love story,music love
7,Avengers,Superhero fights evil,superhero fight
8,John Wick,Assassin revenge,assassin revenge
9,Mad Max,Post-apocalypse chase,post-apocalypse chase


In [112]:
movies_3['combined'] = movies_3['overview'] + ' ' + movies_3['keywords']
movies_3

,movie,overview,keywords,combined
0,Interstellar,Space exploration to save humanity,space nasa future,Space exploration to save humanity space nasa ...
1,Inception,Dream within a dream,dream mind action,Dream within a dream dream mind action
2,The Martian,Astronaut stranded on Mars,mars space survival,Astronaut stranded on Mars mars space survival
3,Gravity,Astronaut in space,space astronaut,Astronaut in space space astronaut
4,Titanic,Love story on a ship,ship love tragedy,Love story on a ship ship love tragedy
5,Notebook,Romantic drama,love relationship,Romantic drama love relationship
6,La La Land,Music and love story,music love,Music and love story music love
7,Avengers,Superhero fights evil,superhero fight,Superhero fights evil superhero fight
8,John Wick,Assassin revenge,assassin revenge,Assassin revenge assassin revenge
9,Mad Max,Post-apocalypse chase,post-apocalypse chase,Post-apocalypse chase post-apocalypse chase


In [113]:
X_new_movies_3 = tfidf.transform(movies_3['combined'])
X_new_movies_3

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 36 stored elements and shape (10, 7064)>

In [114]:
pred_probs_tfidf_3 = model_tfidf.predict_proba(X_new_movies_3)[:,1]
pred_probs_tfidf_3

array([0.48601344, 0.68137673, 0.51616257, 0.52341505, 0.40117123,
       0.42635832, 0.46649732, 0.45458315, 0.60668212, 0.57901083])

In [115]:
movies_3['like_prob_3'] = pred_probs_tfidf_3
movies_3

,movie,overview,keywords,combined,like_prob_3
0,Interstellar,Space exploration to save humanity,space nasa future,Space exploration to save humanity space nasa ...,0.486013
1,Inception,Dream within a dream,dream mind action,Dream within a dream dream mind action,0.681377
2,The Martian,Astronaut stranded on Mars,mars space survival,Astronaut stranded on Mars mars space survival,0.516163
3,Gravity,Astronaut in space,space astronaut,Astronaut in space space astronaut,0.523415
4,Titanic,Love story on a ship,ship love tragedy,Love story on a ship ship love tragedy,0.401171
5,Notebook,Romantic drama,love relationship,Romantic drama love relationship,0.426358
6,La La Land,Music and love story,music love,Music and love story music love,0.466497
7,Avengers,Superhero fights evil,superhero fight,Superhero fights evil superhero fight,0.454583
8,John Wick,Assassin revenge,assassin revenge,Assassin revenge assassin revenge,0.606682
9,Mad Max,Post-apocalypse chase,post-apocalypse chase,Post-apocalypse chase post-apocalypse chase,0.579011


In [ ]:

### Genre Version (Simpler)

print("\n=== Genre Version ===")
###Sample movie dataset with genres
movies_genre = pd.DataFrame({ 'movie': ['Interstellar', 'Inception', 'The Martian', 'Gravity', 'Titanic', 'Notebook', 'La La Land', 'Avengers', 'John Wick', 'Mad Max'], 'SciFi': [1,1,1,1,0,0,0,0,0,0], 'Romance': [0,0,0,0,1,1,1,0,0,0], 'Action': [0,0,0,0,0,0,0,1,1,1] })
#### User likes/dislikes
user_likes_genre = [1,1,1,1,0,0,0,1,1,1]
X_genre = movies_genre[['SciFi','Romance','Action']].values 
model_genre = LogisticRegression() 
model_genre.fit(X_genre, user_likes_genre)
###Predict probabilities
pred_probs_genre = model_genre.predict_proba(X_genre)[:,1] 
movies_genre['like_prob'] = pred_probs_genre 
recommend_genre = movies_genre.sort_values(by='like_prob', ascending=False)[['movie', 'like_prob']] 
print(recommend_genre)